### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/research/failure_regions'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

_data_paths = ExperimentPaths(ROOT, EXPERIMENT_DIR / 'config/paths.json')


In [ ]:
from experiment.utils import misc

def aggregate_path(path):
    return misc.aggregate_path(path, data_paths=_data_paths)


# CA code: BPOSD vs Tesseract

In [ ]:
import multiprocessing
import sys
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import seaborn as sns
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders.cpp_backend.bp_osd import BPOSD
from src.decoders.cpp_backend.tesseract import Tesseract

%load_ext autoreload
%autoreload 2


### Prepare paired decoders and acquisition helpers

The following definitions build the BPOSD/Tesseract comparison, summarize benchmark data and collect outcome-labeled cases. Run them before the noise-model sections, which reuse these helpers with different circuit settings.


In [ ]:
from experiment.utils import decoder_setup

def build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=BPOSD, tesseract_type=Tesseract)


In [ ]:
ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000


#### Collect stats

In [ ]:
def collect_stats(save_file: Path | str, noise_model: str, d_rounds: int=None):
    tasks = []
    custom_decoders = {}
    for code_name, code in ca_codes.items():
        builder = CircuitBuilder(code)
        for p in p_values:
            circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=float(p)), d_rounds=d_rounds)
            bposd, tess = build_decoder_pair(code, float(p))
            b_key = f'{code_name}_BPOSD_p_{p:.6g}'
            t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
            custom_decoders[b_key] = bposd
            custom_decoders[t_key] = tess
            tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
                'code': code_name,
                'decoder': 'BPOSD',
                'p': float(p),
                'noise_model': noise_model,
            }))
            tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
                'code': code_name,
                'decoder': 'TESSERACT',
                'p': float(p),
                'noise_model': noise_model,
            }))

    stats = sinter.collect(
        num_workers=multiprocessing.cpu_count(),
        max_shots=max_shots,
        max_errors=max_errors,
        tasks=tasks,
        decoders=list(custom_decoders.keys()),
        custom_decoders=custom_decoders,
        print_progress=True,
        save_resume_filepath=save_file,
    )
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')

    return stats


In [ ]:
from experiment.utils import benchmark_plotting

stats_to_df = benchmark_plotting.stats_to_df


#### Gap interval

In [ ]:
from experiment.utils import benchmark_plotting

plot_gap_interval = benchmark_plotting.plot_gap_interval


#### Sample Failure Cases

In [ ]:
def bits_to_str(x: np.ndarray) -> str:
    """ord('0')=48"""
    return (x + 48).tobytes().decode('ascii')

def pack_bits_little(arr2d_uint8: np.ndarray) -> np.ndarray:
    return np.packbits(arr2d_uint8.astype(np.uint8), axis=1, bitorder='little')

def unpack_bits_little(arr2d_packed: np.ndarray, n_bits: int) -> np.ndarray:
    return np.unpackbits(arr2d_packed, axis=1, count=n_bits, bitorder='little').astype(np.uint8)

def collect_samples(
    code_name: str,
    ca_code,
    p: float,
    noise_model: str,
    d_rounds: int=None,
    shots: int=1000000000,
    sample_times: int=1000,
    num_samples: int=1000,
    kind: Literal['both_success', 'both_fail', 'bposd_fail_tess_success', 'bposd_success_tess_fail']='bposd_fail_tess_success',
) -> tuple[pd.DataFrame, stim.DetectorErrorModel]:
    """
    Collect cases

    Depending on the noise model and error rate, it can be BPOSD-fail/Tesseract-success cases, or the opposite or both success/fail cases.
    """

    builder = CircuitBuilder(ca_code)
    circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
    dem = circuit.detector_error_model()
    bposd, tess = build_decoder_pair(ca_code, p)
    compiled_b = bposd.compile_decoder_for_dem(dem=dem)
    compiled_t = tess.compile_decoder_for_dem(dem=dem)
    sampler = circuit.compile_detector_sampler()
    num_det = dem.num_detectors
    num_obs = dem.num_observables
    recs = []

    for n in range(sample_times):
        dets, obs = sampler.sample(shots=int(shots), separate_observables=True)
        dets = np.asarray(dets, dtype=np.uint8)
        obs = np.asarray(obs, dtype=np.uint8)
        _, unique_idx = np.unique(dets, axis=0, return_index=True)

        if unique_idx.size == 0:
            continue
        dets = dets[unique_idx]
        obs = obs[unique_idx]
        dets_packed = pack_bits_little(dets)
        pred_b_packed = compiled_b.decode_shots_bit_packed(bit_packed_detection_event_data=dets_packed)
        pred_t_packed = compiled_t.decode_shots_bit_packed(bit_packed_detection_event_data=dets_packed)
        pred_b = unpack_bits_little(np.asarray(pred_b_packed, dtype=np.uint8), num_obs)
        pred_t = unpack_bits_little(np.asarray(pred_t_packed, dtype=np.uint8), num_obs)
        b_succ = np.all(pred_b == obs, axis=1)
        t_succ = np.all(pred_t == obs, axis=1)

        match kind:
            case 'both_success':
                mask = b_succ & t_succ
            case 'both_fail':
                mask = ~b_succ & ~t_succ
            case 'bposd_fail_tess_success':
                mask = ~b_succ & t_succ
            case 'bposd_success_tess_fail':
                mask = b_succ & ~t_succ
            case _:
                raise ValueError(f'Unsupported kind: {kind}')
        idx = np.flatnonzero(mask)

        if idx.size == 0:
            continue
        need = num_samples - len(recs)

        if idx.size > need:
            idx = idx[:need]
        dets_sel = dets[idx]
        obs_sel = obs[idx]
        pred_b_sel = pred_b[idx]
        pred_t_sel = pred_t[idx]
        weights = dets_sel.sum(axis=1)
        recs.extend([{
            'sample_idx': int(n),
            'code': code_name,
            'p': p,
            'shot': int(i),
            'num_detectors': num_det,
            'num_observables': num_obs,
            'detector_weight': int(w),
            'detector_events': bits_to_str(d),
            'true_observables': bits_to_str(o),
            'bposd_pred_observables': bits_to_str(pb),
            'tesseract_pred_observables': bits_to_str(pt),
        } for i, w, d, o, pb, pt in zip(idx, weights, dets_sel, obs_sel, pred_b_sel, pred_t_sel)])

        if len(recs) >= num_samples:
            break

    return (pd.DataFrame(recs), dem)

def collect_samples_with_errors(
    code_name: str,
    ca_code,
    p: float,
    noise_model: str,
    d_rounds: int=None,
    shots: int=1000000000,
    sample_times: int=1000,
    num_samples: int=1000,
    kind: Literal['both_success', 'both_fail', 'bposd_fail_tess_success', 'bposd_success_tess_fail']='bposd_fail_tess_success',
) -> tuple[pd.DataFrame, stim.DetectorErrorModel]:
    """
    Collect cases

    Depending on the noise model and error rate, it can be BPOSD-fail/Tesseract-success cases, or the opposite or both success/fail cases.
    """

    builder = CircuitBuilder(ca_code)
    circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
    dem = circuit.detector_error_model()
    bposd, tess = build_decoder_pair(ca_code, p)
    compiled_b = bposd.compile_decoder_for_dem(dem=dem)
    compiled_t = tess.compile_decoder_for_dem(dem=dem)
    sampler = dem.compile_sampler()
    num_det = dem.num_detectors
    num_obs = dem.num_observables
    recs = []

    for n in range(sample_times):
        dets, obs, errs = sampler.sample(shots=int(shots), return_errors=True)
        dets = np.asarray(dets, dtype=np.uint8)
        obs = np.asarray(obs, dtype=np.uint8)
        errs = np.asarray(errs, dtype=np.uint8)
        _, unique_idx = np.unique(errs, axis=0, return_index=True)

        if unique_idx.size == 0:
            continue
        dets = dets[unique_idx]
        obs = obs[unique_idx]
        errs = errs[unique_idx]
        dets_packed = pack_bits_little(dets)
        pred_b_packed = compiled_b.decode_shots_bit_packed(bit_packed_detection_event_data=dets_packed)
        pred_t_packed = compiled_t.decode_shots_bit_packed(bit_packed_detection_event_data=dets_packed)
        pred_b = unpack_bits_little(np.asarray(pred_b_packed, dtype=np.uint8), num_obs)
        pred_t = unpack_bits_little(np.asarray(pred_t_packed, dtype=np.uint8), num_obs)
        b_succ = np.all(pred_b == obs, axis=1)
        t_succ = np.all(pred_t == obs, axis=1)

        match kind:
            case 'both_success':
                mask = b_succ & t_succ
            case 'both_fail':
                mask = ~b_succ & ~t_succ
            case 'bposd_fail_tess_success':
                mask = ~b_succ & t_succ
            case 'bposd_success_tess_fail':
                mask = b_succ & ~t_succ
            case _:
                raise ValueError(f'Unsupported kind: {kind}')
        idx = np.flatnonzero(mask)

        if idx.size == 0:
            continue
        need = num_samples - len(recs)

        if idx.size > need:
            idx = idx[:need]
        dets_sel = dets[idx]
        obs_sel = obs[idx]
        errs_sel = errs[idx]
        pred_b_sel = pred_b[idx]
        pred_t_sel = pred_t[idx]
        det_weights = dets_sel.sum(axis=1)
        err_weights = errs_sel.sum(axis=1)
        recs.extend([{
            'sample_idx': int(n),
            'code': code_name,
            'p': p,
            'shot': int(i),
            'num_detectors': num_det,
            'num_observables': num_obs,
            'detector_weight': int(dw),
            'detector_events': bits_to_str(d),
            'error_events': bits_to_str(e),
            'error_weight': int(ew),
            'true_observables': bits_to_str(o),
            'bposd_pred_observables': bits_to_str(pb),
            'tesseract_pred_observables': bits_to_str(pt),
        } for i, dw, ew, d, o, e, pb, pt in zip(idx, det_weights, err_weights, dets_sel, obs_sel, errs_sel, pred_b_sel, pred_t_sel)])

        if len(recs) >= num_samples:
            break

    return (pd.DataFrame(recs), dem)


In [ ]:
def count_samples_union(
    code_name: str,
    ca_code,
    p: float,
    noise_model: str,
    d_rounds: int=None,
    shots: int=1000000000,
    sample_times: int=1000,
    num_samples: int=1000,
) -> tuple[int, int, int, int, stim.DetectorErrorModel]:
    """
    Count samples in all different kinds
    """

    builder = CircuitBuilder(ca_code)
    circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
    dem = circuit.detector_error_model()
    bposd, tess = build_decoder_pair(ca_code, p)
    compiled_b = bposd.compile_decoder_for_dem(dem=dem)
    compiled_t = tess.compile_decoder_for_dem(dem=dem)
    sampler = circuit.compile_detector_sampler()
    num_det = dem.num_detectors
    num_obs = dem.num_observables
    bs_cnt, bf_cnt, bfts_cnt, bstf_cnt = (0, 0, 0, 0)

    for n in range(sample_times):
        dets, obs = sampler.sample(shots=int(shots), separate_observables=True)
        dets = np.asarray(dets, dtype=np.uint8)
        obs = np.asarray(obs, dtype=np.uint8)
        _, unique_idx = np.unique(dets, axis=0, return_index=True)

        if unique_idx.size == 0:
            continue
        dets = dets[unique_idx]
        obs = obs[unique_idx]
        dets_packed = pack_bits_little(dets)
        pred_b_packed = compiled_b.decode_shots_bit_packed(bit_packed_detection_event_data=dets_packed)
        pred_t_packed = compiled_t.decode_shots_bit_packed(bit_packed_detection_event_data=dets_packed)
        pred_b = unpack_bits_little(np.asarray(pred_b_packed, dtype=np.uint8), num_obs)
        pred_t = unpack_bits_little(np.asarray(pred_t_packed, dtype=np.uint8), num_obs)
        b_succ = np.all(pred_b == obs, axis=1)
        t_succ = np.all(pred_t == obs, axis=1)
        bs_cnt += np.sum(b_succ & t_succ)
        bf_cnt += np.sum(~b_succ & ~t_succ)
        bfts_cnt += np.sum(~b_succ & t_succ)
        bstf_cnt += np.sum(b_succ & ~t_succ)
        total_fails = bf_cnt + bfts_cnt + bstf_cnt

        if total_fails >= num_samples:
            break

    return (bs_cnt, bf_cnt, bfts_cnt, bstf_cnt, dem)


# Code-Capacity

### Code-capacity screening

Read or collect the code-capacity benchmark, identify the selected gap regimes and retain disagreement samples. The sampled case table is a diagnostic selection rather than an unbiased estimate from its retained row counts.


In [ ]:
DATA_DIR = _data_paths.output('code_capacity')
DATA_DIR.mkdir(parents=True, exist_ok=True)
save_file = _data_paths.output('ca_sinter_bposd_tesseract_csv', path_component=DATA_DIR.name)


In [ ]:
stats = collect_stats(save_file=save_file, noise_model='code_capacity')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
df = stats_to_df(stats)
summary = df.pivot_table(
    index=['code', 'p'],
    columns='decoder',
    values='logical_error_rate',
    aggfunc='mean',
).reset_index().rename_axis(None, axis=1)
summary['gap_bposd_minus_tess'] = summary['BPOSD'] - summary['TESSERACT']
summary = summary.sort_values(['code', 'p']).reset_index(drop=True)
summary


In [ ]:
min_bposd_ler = 0
min_gap = 0
failure_region = summary[(summary['BPOSD'] >= min_bposd_ler) & (summary['gap_bposd_minus_tess'] >= min_gap)].copy()
failure_region


## Gap interval
The BPOSD-Tesseract gap

In [ ]:
plot_gap_interval(summary, x_scale=('log', 'linear'), y_scale=('log', 'linear'))


In [ ]:

from experiment.utils.benchmark_plotting import extract_gap_intervals
gap_tables = []
for code_name in sorted(summary['code'].unique()):
    d = summary[summary['code'] == code_name].copy()
    d['gap_bposd_minus_tess'] = d['BPOSD'] - d['TESSERACT']
    tab = extract_gap_intervals(d, min_gap=0.0)

    if len(tab) > 0:
        tab.insert(0, 'code', code_name)
        gap_tables.append(tab)

gap_intervals = pd.concat(gap_tables, ignore_index=True) if len(gap_tables) > 0 else pd.DataFrame()
gap_intervals


## Sample failures on gap intervals
This section collects detector-event error samples where BPOSD fails but Tesseract succeeds

In [ ]:
CASES_DIR = _data_paths.output('cases')
CASES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
min_gap = 0
points = summary.copy()
points = points[points['gap_bposd_minus_tess'] > min_gap].sort_values(['code', 'p'])
points


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model='code_capacity',
        p=p,
        shots=5000,
    )

    print(f'collected {len(cdf)} bposd_fail_tess_success cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

failure_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bfts_cases_csv')
if not failure_cases.empty:
    failure_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
failure_cases.head(20)


In [ ]:
failure_cases['code'].value_counts()


# Phenomenological

### Phenomenological screening

Repeat the benchmark/gap/case workflow under phenomenological noise. Keep the configured round count and this model's paths with its results rather than combining them with code-capacity cases.


In [ ]:
DATA_DIR = _data_paths.output('phenomenological')
DATA_DIR.mkdir(parents=True, exist_ok=True)
save_file = _data_paths.output('ca_sinter_bposd_tesseract_csv', path_component=DATA_DIR.name)


In [ ]:
stats = collect_stats(save_file=save_file, noise_model='phenomenological', d_rounds=3)


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
df = stats_to_df(stats)
summary = df.pivot_table(
    index=['code', 'p'],
    columns='decoder',
    values='logical_error_rate',
    aggfunc='mean',
).reset_index().rename_axis(None, axis=1)
summary['gap_bposd_minus_tess'] = summary['BPOSD'] - summary['TESSERACT']
summary = summary.sort_values(['code', 'p']).reset_index(drop=True)
summary


In [ ]:
min_bposd_ler = 5e-10
min_gap = 2e-10
failure_region = summary[(summary['BPOSD'] >= min_bposd_ler) & (summary['gap_bposd_minus_tess'] >= min_gap)].copy()
failure_region


## Gap interval
The BPOSD-Tesseract gap

In [ ]:
plot_gap_interval(summary, x_scale=('log', 'linear'), y_scale=('log', 'linear'))


## Sample failures on gap intervals
This section collects detector-event error samples where BPOSD fails but Tesseract succeeds

In [ ]:
CASES_DIR = _data_paths.output('cases_1')
CASES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
min_gap = 0
points = summary.copy()
points = points[points['gap_bposd_minus_tess'] > min_gap].sort_values(['code', 'p'])
points


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model='phenomenological',
        p=p,
        d_rounds=3,
        shots=5000,
    )

    print(f'collected {len(cdf)} failure cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bfts_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bfts_cases_csv_2')
if not bfts_cases.empty:
    bfts_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bfts_cases.head(20)


In [ ]:
bfts_cases['code'].value_counts()


sample the control cases

1. both success
2. bposd_success_tess_fail

### Acquire the control outcome groups

Collect both-success, reverse-disagreement and both-failure examples alongside the target disagreement. Later comparisons need these controls to distinguish a failure-associated feature from a common graph property.


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model='phenomenological',
        d_rounds=3,
        kind='both_success',
        shots=5000,
    )

    print(f'collected {len(cdf)} both_success cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bs_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bs_cases_csv')
if not bs_cases.empty:
    bs_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bs_cases.head(20)


In [ ]:
bs_cases['code'].value_counts()


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model='phenomenological',
        d_rounds=3,
        kind='bposd_success_tess_fail',
        shots=5000,
    )

    print(f'collected {len(cdf)} bposd_success_tess_fail cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bstf_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bstf_cases_csv')
if not bstf_cases.empty:
    bstf_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bstf_cases.head(20)


In [ ]:
bstf_cases['code'].value_counts()


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model='phenomenological',
        d_rounds=3,
        kind='both_fail',
        shots=5000,
    )

    print(f'collected {len(cdf)} both_fail cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bf_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bf_cases_csv')
if not bf_cases.empty:
    bf_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bf_cases.head(20)


In [ ]:
bf_cases['code'].value_counts()


In [ ]:
bf_cases = pd.read_csv(_data_paths.output('ca_bf_cases_csv'))
bs_cases = pd.read_csv(_data_paths.output('ca_bs_cases_csv'))
bfts_cases = pd.read_csv(_data_paths.output('ca_bfts_cases_csv_3'))
bstf_cases = pd.read_csv(_data_paths.output('ca_bstf_cases_csv'))


In [ ]:
frac_bp = bf_cases.shape[0] / (bf_cases.shape[0] + bfts_cases.shape[0])
frac_tess = bf_cases.shape[0] / (bf_cases.shape[0] + bstf_cases.shape[0])
print(f'fraction of both fail cases among all bp fail samples: {frac_bp:.4f}')
print(f'fraction of both fail cases among all tess fail samples: {frac_tess:.4f}')


### Counting

### Count paired outcomes

The counting block measures all four decoder outcomes under the stated acquisition budget and summarizes their overlap. Distinguish these shot-level counts from selected or capped example tables.


In [ ]:
shots = 1000000
d_rounds = 3
noise_model = 'phenomenological'
cases_path = aggregate_path(_data_paths.output('cnt_cases_noise_model_d_rounds_shots_shots_csv', noise_model=noise_model, d_rounds=d_rounds, shots=shots))


In [ ]:
f = open(cases_path, 'w')
f.write('code,p,both_success,both_failure,bposd_fail_tess_success,bposd_success_tess_fail\n')
bs_total_cnt, bf_total_cnt, bfts_total_cnt, bstf_total_cnt = (0, 0, 0, 0)
cnt_table = []

for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'counting cases: code={code_name}, p={p:.6f}')
    bs_cnt, bf_cnt, bfts_cnt, bstf_cnt, dem = count_samples_union(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        p=p,
        d_rounds=d_rounds,
        shots=shots,
    )

    print(f'counted {bs_cnt} both success cases for code={code_name}, p={p:.6f}')
    print(f'counted {bf_cnt} both failure cases for code={code_name}, p={p:.6f}')
    print(f'counted {bfts_cnt} failure success cases for code={code_name}, p={p:.6f}')
    print(f'counted {bstf_cnt} success failure cases for code={code_name}, p={p:.6f}')
    print('=================================')
    bs_total_cnt += bs_cnt
    bf_total_cnt += bf_cnt
    bfts_total_cnt += bfts_cnt
    bstf_total_cnt += bstf_cnt
    f.write(f'{code_name},{p},{bs_cnt},{bf_cnt},{bfts_cnt},{bstf_cnt}\n')
    f.flush()

f.close()
print(f'saved: {cases_path}')


In [ ]:
frac_bp = bf_total_cnt / (bf_total_cnt + bfts_total_cnt)
frac_tess = bf_total_cnt / (bf_total_cnt + bstf_total_cnt)
print(f'fraction of both fail cases among all bp fail samples: {frac_bp:.4f}')
print(f'fraction of both fail cases among all tess fail samples: {frac_tess:.4f}')


In [ ]:
cnt_data = pd.read_csv(cases_path)
pivot_table = cnt_data.pivot(index='p', columns='code', values=['both_success', 'both_failure', 'bposd_fail_tess_success', 'bposd_success_tess_fail'])
pivot_table


In [ ]:
frac = []
for p, row in pivot_table.iterrows():
    for code_name in ca_codes.keys():
        frac_bp = row['both_failure'][code_name] / (row['both_failure'][code_name] + row['bposd_fail_tess_success'][code_name])
        frac_tess = row['both_failure'][code_name] / (row['both_failure'][code_name] + row['bposd_success_tess_fail'][code_name])
        frac.append({
            'code': code_name,
            'p': p,
            'frac_bp': frac_bp,
            'frac_tess': frac_tess,
        })

frac_df = pd.DataFrame(frac).sort_values(['code', 'p']).reset_index(drop=True)
frac_df


In [ ]:
plt.figure(figsize=(10, 6))
for code_name in ca_codes.keys():
    sub = frac_df[frac_df['code'] == code_name]
    plt.plot(sub['p'], sub['frac_bp'], marker='o', label=f'frac_bp ({code_name})')
    plt.plot(sub['p'], sub['frac_tess'], marker='s', label=f'frac_tess ({code_name})')
    plt.xlabel('p')
    plt.ylabel('Fraction')
    plt.title('Failure Rates by Region')
    plt.legend()

plt.show()


In [ ]:
fig, axes = plt.subplots(2, 9, figsize=(48, 18))
fig.suptitle('Overlap between BPOSD Failure and Tesseract Failure cases', fontsize=24)
axes = axes.flatten()

for i, row in cnt_data.iterrows():
    frac_bp = row['both_failure'] / (row['both_failure'] + row['bposd_fail_tess_success'])
    frac_tess = row['both_failure'] / (row['both_failure'] + row['bposd_success_tess_fail'])
    venn2(subsets=(row['bposd_fail_tess_success'], row['bposd_success_tess_fail'], row['both_failure']), set_labels=(f'BPOSD Fails \n({frac_bp:.2%})', f'Tesseract Fails \n({frac_tess:.2%})'), ax=axes[i])
    axes[i].set_title(f"{row['code']} (p={row['p']:.2})", fontsize=16)

plt.tight_layout()
plt.show()


# Circuit-level

### Circuit-level screening

Repeat the paired comparison with circuit-level noise and the configured rounds. Use this section's benchmark and sample paths when interpreting its disagreement populations.


In [ ]:
DATA_DIR = _data_paths.output('circuit_level')
DATA_DIR.mkdir(parents=True, exist_ok=True)
save_file = _data_paths.output('ca_sinter_bposd_tesseract_csv', path_component=DATA_DIR.name)


In [ ]:
stats = collect_stats(save_file=save_file, noise_model='circuit_level', d_rounds=3)


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
df = stats_to_df(stats)
summary = df.pivot_table(
    index=['code', 'p'],
    columns='decoder',
    values='logical_error_rate',
    aggfunc='mean',
).reset_index().rename_axis(None, axis=1)
summary['gap_bposd_minus_tess'] = summary['BPOSD'] - summary['TESSERACT']
summary = summary.sort_values(['code', 'p']).reset_index(drop=True)
summary


In [ ]:
min_bposd_ler = 5e-10
min_gap = 2e-10
failure_region = summary[(summary['BPOSD'] >= min_bposd_ler) & (summary['gap_bposd_minus_tess'] >= min_gap)].copy()
failure_region


## Gap interval
The BPOSD-Tesseract gap

In [ ]:
plot_gap_interval(summary, x_scale=('log', 'log'), y_scale=('log', 'log'))


In [ ]:

from experiment.utils.benchmark_plotting import extract_gap_intervals
gap_tables = []
for code_name in sorted(summary['code'].unique()):
    d = summary[summary['code'] == code_name].copy()
    d['gap_bposd_minus_tess'] = d['BPOSD'] - d['TESSERACT']
    tab = extract_gap_intervals(d, min_gap=0.0)

    if len(tab) > 0:
        tab.insert(0, 'code', code_name)
        gap_tables.append(tab)

gap_intervals = pd.concat(gap_tables, ignore_index=True) if len(gap_tables) > 0 else pd.DataFrame()
gap_intervals


## Sample failures on gap intervals
This section collects detector-event error samples where BPOSD fails but Tesseract succeeds

In [ ]:
CASES_DIR = _data_paths.output('cases_4')
CASES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
min_gap = 0
points = summary.copy()
points = points[points['gap_bposd_minus_tess'] > min_gap].sort_values(['code', 'p'])
points


In [ ]:
shots = 1000000
d_rounds = 3
noise_model = 'circuit_level'


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        p=p,
        d_rounds=d_rounds,
        shots=shots,
    )

    print(f'collected {len(cdf)} failure cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bfts_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bfts_cases_csv_5')
if not bfts_cases.empty:
    bfts_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bfts_cases.head(20)


In [ ]:
bfts_cases['code'].value_counts()


sample the control cases

1. both success
2. bposd_success_tess_fail

In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model=noise_model,
        d_rounds=d_rounds,
        kind='both_success',
        shots=shots,
    )

    print(f'collected {len(cdf)} both_success cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bs_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bs_cases_csv_6')
if not bs_cases.empty:
    bs_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bs_cases.head(20)


In [ ]:
bs_cases['code'].value_counts()


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model=noise_model,
        d_rounds=d_rounds,
        kind='bposd_success_tess_fail',
        shots=shots,
    )

    print(f'collected {len(cdf)} bposd_success_tess_fail cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bstf_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bstf_cases_csv_7')
if not bstf_cases.empty:
    bstf_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bstf_cases.head(20)


In [ ]:
bstf_cases['code'].value_counts()


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model=noise_model,
        d_rounds=d_rounds,
        kind='both_fail',
        shots=shots,
    )

    print(f'collected {len(cdf)} both_fail cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bf_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bf_cases_csv_8')
if not bf_cases.empty:
    bf_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bf_cases.head(20)


In [ ]:
bf_cases['code'].value_counts()


In [ ]:
bf_cases = pd.read_csv(_data_paths.output('ca_bf_cases_csv_9'))
bs_cases = pd.read_csv(_data_paths.output('ca_bs_cases_csv_10'))
bfts_cases = pd.read_csv(_data_paths.output('ca_bfts_cases_csv_11'))
bstf_cases = pd.read_csv(_data_paths.output('ca_bstf_cases_csv_12'))


In [ ]:
frac_bp = bf_cases.shape[0] / (bf_cases.shape[0] + bfts_cases.shape[0])
frac_tess = bf_cases.shape[0] / (bf_cases.shape[0] + bstf_cases.shape[0])
print(f'fraction of both fail cases among all bp fail samples: {frac_bp:.4f}')
print(f'fraction of both fail cases among all tess fail samples: {frac_tess:.4f}')


### Counting

### Summarize circuit-level overlap

Count and visualize the joint decoder outcomes for the selected circuit-level settings. Interpret overlap at each code and p instead of inferring an overall ranking from pooled retained examples.


In [ ]:
shots = 1000000
d_rounds = 3
noise_model = 'circuit_level'
cases_path = aggregate_path(_data_paths.output('cnt_cases_noise_model_d_rounds_shots_shots_csv_13', noise_model=noise_model, d_rounds=d_rounds, shots=shots))


In [ ]:
f = open(cases_path, 'w')
f.write('code,p,both_success,both_failure,bposd_fail_tess_success,bposd_success_tess_fail\n')
bs_total_cnt, bf_total_cnt, bfts_total_cnt, bstf_total_cnt = (0, 0, 0, 0)
cnt_table = []

for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'counting cases: code={code_name}, p={p:.6f}')
    bs_cnt, bf_cnt, bfts_cnt, bstf_cnt, dem = count_samples_union(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        p=p,
        d_rounds=d_rounds,
        shots=shots,
    )

    print(f'counted {bs_cnt} both success cases for code={code_name}, p={p:.6f}')
    print(f'counted {bf_cnt} both failure cases for code={code_name}, p={p:.6f}')
    print(f'counted {bfts_cnt} failure success cases for code={code_name}, p={p:.6f}')
    print(f'counted {bstf_cnt} success failure cases for code={code_name}, p={p:.6f}')
    print('=================================')
    bs_total_cnt += bs_cnt
    bf_total_cnt += bf_cnt
    bfts_total_cnt += bfts_cnt
    bstf_total_cnt += bstf_cnt
    f.write(f'{code_name},{p},{bs_cnt},{bf_cnt},{bfts_cnt},{bstf_cnt}\n')
    f.flush()

f.close()
print(f'saved: {cases_path}')


In [ ]:
frac_bp = bf_total_cnt / (bf_total_cnt + bfts_total_cnt)
frac_tess = bf_total_cnt / (bf_total_cnt + bstf_total_cnt)
print(f'fraction of both fail cases among all bp fail samples: {frac_bp:.4f}')
print(f'fraction of both fail cases among all tess fail samples: {frac_tess:.4f}')


In [ ]:
cnt_data = pd.read_csv(cases_path)
pivot_table = cnt_data.pivot(index='p', columns='code', values=['both_success', 'both_failure', 'bposd_fail_tess_success', 'bposd_success_tess_fail'])
pivot_table


In [ ]:
frac = []
for p, row in pivot_table.iterrows():
    if row.isnull().any():
        continue
    for code_name in ca_codes.keys():
        total_bp_fail = row['both_failure'][code_name] + row['bposd_fail_tess_success'][code_name]
        total_tess_fail = row['both_failure'][code_name] + row['bposd_success_tess_fail'][code_name]
        if total_bp_fail == 0 or total_tess_fail == 0:
            continue
        frac_bp = row['both_failure'][code_name] / total_bp_fail
        frac_tess = row['both_failure'][code_name] / total_tess_fail
        frac.append({
            'code': code_name,
            'p': p,
            'frac_bp': frac_bp,
            'frac_tess': frac_tess,
        })

frac_df = pd.DataFrame(frac).sort_values(['code', 'p']).reset_index(drop=True)
frac_df


In [ ]:
plt.figure(figsize=(10, 6))
for code_name in ca_codes.keys():
    sub = frac_df[frac_df['code'] == code_name]
    plt.plot(sub['p'], sub['frac_bp'], marker='o', label=f'frac_bp ({code_name})')
    plt.plot(sub['p'], sub['frac_tess'], marker='s', label=f'frac_tess ({code_name})')
    plt.xlabel('p')
    plt.ylabel('Fraction')
    plt.title('Failure Rates by Region')
    plt.legend()

plt.show()


In [ ]:
fig, axes = plt.subplots(2, cnt_data.shape[0] // 2, figsize=(48, 18))
fig.suptitle('Overlap between BPOSD Failure and Tesseract Failure cases', fontsize=24)
axes = axes.flatten()

for i, row in cnt_data.iterrows():
    if row.isnull().any():
        continue
    total_bp_fail = row['both_failure'] + row['bposd_fail_tess_success']
    total_tess_fail = row['both_failure'] + row['bposd_success_tess_fail']

    if total_bp_fail == 0 or total_tess_fail == 0:
        continue
    frac_bp = row['both_failure'] / total_bp_fail
    frac_tess = row['both_failure'] / total_tess_fail
    venn2(subsets=(row['bposd_fail_tess_success'], row['bposd_success_tess_fail'], row['both_failure']), set_labels=(f'BPOSD Fails \n({frac_bp:.2%})', f'Tesseract Fails \n({frac_tess:.2%})'), ax=axes[i])
    axes[i].set_title(f"{row['code']} (p={row['p']:.2})", fontsize=16)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, cnt_data.shape[0] // 2, figsize=(48, 18))
fig.suptitle('Confusion Matrix for Success/Failure cases', fontsize=24)
axes = axes.flatten()

for i, row in cnt_data.iterrows():
    if row.isnull().any():
        continue
    data = np.array([[row['both_failure'], row['bposd_fail_tess_success']], [row['bposd_success_tess_fail'], row['both_success']]])

    im = axes[i].imshow(data, cmap='Blues')
    for j in range(2):
        for k in range(2):
            axes[i].text(k, j, str(data[j, k]), ha='center', va='center', fontsize=16)

    axes[i].set_xticks([0, 1])
    axes[i].set_yticks([0, 1])
    axes[i].set_xticklabels(['Tesseract Fails', 'Tesseract Success'])
    axes[i].set_yticklabels(['BPOSD Fails', 'BPOSD Success'])
    axes[i].set_title(f"{row['code']} (p={row['p']:.2})", fontsize=16)

plt.tight_layout()
plt.show()


## One round circuit-level noise except for the measurements which are perfect

### One round with perfect measurement

The next experiment changes the circuit to a one-round, perfect-measurement setting. It has its own benchmark and case-acquisition blocks and should not be pooled with the multi-round experiment above.


In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
save_file = _data_paths.output(
    'noise_model_d_rounds_perfect_meas_shots_max_shots_csv',
    path_component=DATA_DIR.name,
    noise_model=noise_model,
    d_rounds=d_rounds,
    max_shots=max_shots,
)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(
            noise_model=noise_model,
            error_rate=ErrorRate(p=float(p), p_meas=0),
            d_rounds=d_rounds,
        )
        bposd, tess = build_decoder_pair(code, float(p))
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=True,
    save_resume_filepath=save_file,
)
print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
df = stats_to_df(stats)
summary = df.pivot_table(
    index=['code', 'p'],
    columns='decoder',
    values='logical_error_rate',
    aggfunc='mean',
).reset_index().rename_axis(None, axis=1)
summary['gap_bposd_minus_tess'] = summary['BPOSD'] - summary['TESSERACT']
summary = summary.sort_values(['code', 'p']).reset_index(drop=True)


In [ ]:
plot_gap_interval(summary, x_scale=('log', 'log'), y_scale=('log', 'log'))


## Sample failures on gap intervals
This section collects detector-event error samples where BPOSD fails but Tesseract succeeds

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
shots = 1000000
CASES_DIR = _data_paths.output('noise_model_d_rounds_perfect_meas_shots_shots', noise_model=noise_model, d_rounds=d_rounds, shots=shots)
CASES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
min_gap = 0
points = summary.copy()
points = points[points['gap_bposd_minus_tess'] > min_gap].sort_values(['code', 'p'])
points


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        p=p,
        d_rounds=d_rounds,
        shots=shots,
    )

    print(f'collected {len(cdf)} failure cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bfts_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bfts_cases_csv', noise_model=noise_model, d_rounds=d_rounds, shots=shots)
if not bfts_cases.empty:
    bfts_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bfts_cases.head(20)


In [ ]:
bfts_cases['code'].value_counts()


sample the control cases

1. both success
2. bposd_success_tess_fail

In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model=noise_model,
        d_rounds=d_rounds,
        kind='both_success',
        shots=shots,
    )

    print(f'collected {len(cdf)} both_success cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bs_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bs_cases_csv', noise_model=noise_model, d_rounds=d_rounds, shots=shots)
if not bs_cases.empty:
    bs_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bs_cases.head(20)


In [ ]:
bs_cases['code'].value_counts()


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model=noise_model,
        d_rounds=d_rounds,
        kind='bposd_success_tess_fail',
        shots=shots,
    )

    print(f'collected {len(cdf)} bposd_success_tess_fail cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bstf_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bstf_cases_csv', noise_model=noise_model, d_rounds=d_rounds, shots=shots)
if not bstf_cases.empty:
    bstf_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bstf_cases.head(20)


In [ ]:
bstf_cases['code'].value_counts()


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model=noise_model,
        d_rounds=d_rounds,
        kind='both_fail',
        shots=shots,
    )

    print(f'collected {len(cdf)} both_fail cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bf_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bf_cases_csv', noise_model=noise_model, d_rounds=d_rounds, shots=shots)
if not bf_cases.empty:
    bf_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bf_cases.head(20)


In [ ]:
bf_cases['code'].value_counts()


In [ ]:
bf_cases = pd.read_csv(_data_paths.output('ca_bf_cases_csv', noise_model=noise_model, d_rounds=d_rounds, shots=shots))
bs_cases = pd.read_csv(_data_paths.output('ca_bs_cases_csv', noise_model=noise_model, d_rounds=d_rounds, shots=shots))
bfts_cases = pd.read_csv(_data_paths.output('ca_bfts_cases_csv', noise_model=noise_model, d_rounds=d_rounds, shots=shots))
bstf_cases = pd.read_csv(_data_paths.output('ca_bstf_cases_csv', noise_model=noise_model, d_rounds=d_rounds, shots=shots))


In [ ]:
frac_bp = bf_cases.shape[0] / (bf_cases.shape[0] + bfts_cases.shape[0])
frac_tess = bf_cases.shape[0] / (bf_cases.shape[0] + bstf_cases.shape[0])
print(f'fraction of both fail cases among all bp fail samples: {frac_bp:.4f}')
print(f'fraction of both fail cases among all tess fail samples: {frac_tess:.4f}')


### Counting

In [ ]:
shots = 1000000
d_rounds = 3
noise_model = 'circuit_level'
cases_path = aggregate_path(_data_paths.output('cnt_cases_noise_model_d_rounds_shots_shots_csv_14', noise_model=noise_model, d_rounds=d_rounds, shots=shots))


In [ ]:
f = open(cases_path, 'w')
f.write('code,p,both_success,both_failure,bposd_fail_tess_success,bposd_success_tess_fail\n')
bs_total_cnt, bf_total_cnt, bfts_total_cnt, bstf_total_cnt = (0, 0, 0, 0)
cnt_table = []

for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'counting cases: code={code_name}, p={p:.6f}')
    bs_cnt, bf_cnt, bfts_cnt, bstf_cnt, dem = count_samples_union(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        p=p,
        d_rounds=d_rounds,
        shots=shots,
    )

    print(f'counted {bs_cnt} both success cases for code={code_name}, p={p:.6f}')
    print(f'counted {bf_cnt} both failure cases for code={code_name}, p={p:.6f}')
    print(f'counted {bfts_cnt} failure success cases for code={code_name}, p={p:.6f}')
    print(f'counted {bstf_cnt} success failure cases for code={code_name}, p={p:.6f}')
    print('=================================')
    bs_total_cnt += bs_cnt
    bf_total_cnt += bf_cnt
    bfts_total_cnt += bfts_cnt
    bstf_total_cnt += bstf_cnt
    f.write(f'{code_name},{p},{bs_cnt},{bf_cnt},{bfts_cnt},{bstf_cnt}\n')
    f.flush()

f.close()
print(f'saved: {cases_path}')


In [ ]:
frac_bp = bf_total_cnt / (bf_total_cnt + bfts_total_cnt)
frac_tess = bf_total_cnt / (bf_total_cnt + bstf_total_cnt)
print(f'fraction of both fail cases among all bp fail samples: {frac_bp:.4f}')
print(f'fraction of both fail cases among all tess fail samples: {frac_tess:.4f}')


In [ ]:
cnt_data = pd.read_csv(cases_path)
pivot_table = cnt_data.pivot(index='p', columns='code', values=['both_success', 'both_failure', 'bposd_fail_tess_success', 'bposd_success_tess_fail'])
pivot_table


In [ ]:
frac = []
for p, row in pivot_table.iterrows():
    if row.isnull().any():
        continue
    for code_name in ca_codes.keys():
        total_bp_fail = row['both_failure'][code_name] + row['bposd_fail_tess_success'][code_name]
        total_tess_fail = row['both_failure'][code_name] + row['bposd_success_tess_fail'][code_name]
        if total_bp_fail == 0 or total_tess_fail == 0:
            continue
        frac_bp = row['both_failure'][code_name] / total_bp_fail
        frac_tess = row['both_failure'][code_name] / total_tess_fail
        frac.append({
            'code': code_name,
            'p': p,
            'frac_bp': frac_bp,
            'frac_tess': frac_tess,
        })

frac_df = pd.DataFrame(frac).sort_values(['code', 'p']).reset_index(drop=True)
frac_df


In [ ]:
plt.figure(figsize=(10, 6))
for code_name in ca_codes.keys():
    sub = frac_df[frac_df['code'] == code_name]
    plt.plot(sub['p'], sub['frac_bp'], marker='o', label=f'frac_bp ({code_name})')
    plt.plot(sub['p'], sub['frac_tess'], marker='s', label=f'frac_tess ({code_name})')
    plt.xlabel('p')
    plt.ylabel('Fraction')
    plt.title('Failure Rates by Region')
    plt.legend()

plt.show()


In [ ]:
fig, axes = plt.subplots(2, (cnt_data.shape[0] + 1) // 2, figsize=(48, 18))
fig.suptitle('Overlap between BPOSD Failure and Tesseract Failure cases', fontsize=24)
axes = axes.flatten()

for i, row in cnt_data.iterrows():
    if row.isnull().any():
        continue
    total_bp_fail = row['both_failure'] + row['bposd_fail_tess_success']
    total_tess_fail = row['both_failure'] + row['bposd_success_tess_fail']

    if total_bp_fail == 0 or total_tess_fail == 0:
        continue
    frac_bp = row['both_failure'] / total_bp_fail
    frac_tess = row['both_failure'] / total_tess_fail
    venn2(subsets=(row['bposd_fail_tess_success'], row['bposd_success_tess_fail'], row['both_failure']), set_labels=(f'BPOSD Fails \n({frac_bp:.2%})', f'Tesseract Fails \n({frac_tess:.2%})'), ax=axes[i])
    axes[i].set_title(f"{row['code']} (p={row['p']:.2})", fontsize=16)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, (cnt_data.shape[0] + 1) // 2, figsize=(48, 18))
fig.suptitle('Confusion Matrix for Success/Failure cases', fontsize=24)
axes = axes.flatten()

for i, row in cnt_data.iterrows():
    if row.isnull().any():
        continue
    data = np.array([[row['both_failure'], row['bposd_fail_tess_success']], [row['bposd_success_tess_fail'], row['both_success']]])

    im = axes[i].imshow(data, cmap='Blues')
    for j in range(2):
        for k in range(2):
            axes[i].text(k, j, str(data[j, k]), ha='center', va='center', fontsize=16)

    axes[i].set_xticks([0, 1])
    axes[i].set_yticks([0, 1])
    axes[i].set_xticklabels(['Tesseract Fails', 'Tesseract Success'])
    axes[i].set_yticklabels(['BPOSD Fails', 'BPOSD Success'])
    axes[i].set_title(f"{row['code']} (p={row['p']:.2})", fontsize=16)

plt.tight_layout()
plt.show()


### Sample with errors

### Retain physical error explanations

Collect cases with the sampled error support as well as detectors and logical outcomes. These records support downstream trapping-set and error-weight analyses; they contain offline truth unavailable to the decoder.


In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
shots = 1000000
CASES_DIR = _data_paths.output('with_errors', noise_model=noise_model, d_rounds=d_rounds, shots=shots)
CASES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
min_gap = 0
points = summary.copy()
points = points[points['gap_bposd_minus_tess'] > min_gap].sort_values(['code', 'p'])
points


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        p=p,
        d_rounds=d_rounds,
        shots=shots,
    )

    print(f'collected {len(cdf)} failure cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bfts_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bfts_cases_csv_15', noise_model=noise_model, d_rounds=d_rounds, shots=shots)
if not bfts_cases.empty:
    bfts_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bfts_cases.head(20)


In [ ]:
bfts_cases['code'].value_counts()


sample the control cases

1. both success
2. bposd_success_tess_fail

In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model=noise_model,
        d_rounds=d_rounds,
        kind='both_success',
        shots=shots,
    )

    print(f'collected {len(cdf)} both_success cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bs_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bs_cases_csv_16', noise_model=noise_model, d_rounds=d_rounds, shots=shots)
if not bs_cases.empty:
    bs_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bs_cases.head(20)


In [ ]:
bs_cases['code'].value_counts()


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model=noise_model,
        d_rounds=d_rounds,
        kind='bposd_success_tess_fail',
        shots=shots,
    )

    print(f'collected {len(cdf)} bposd_success_tess_fail cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bstf_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bstf_cases_csv_17', noise_model=noise_model, d_rounds=d_rounds, shots=shots)
if not bstf_cases.empty:
    bstf_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bstf_cases.head(20)


In [ ]:
bstf_cases['code'].value_counts()


In [ ]:
cases_all = []
for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    cdf, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        p=p,
        noise_model=noise_model,
        d_rounds=d_rounds,
        kind='both_fail',
        shots=shots,
    )

    print(f'collected {len(cdf)} both_fail cases for code={code_name}, p={p:.6f}')
    cases_all.append(cdf)

bf_cases = pd.concat(cases_all, ignore_index=True) if len(cases_all) > 0 else pd.DataFrame()
cases_path = _data_paths.output('ca_bf_cases_csv_18', noise_model=noise_model, d_rounds=d_rounds, shots=shots)
if not bf_cases.empty:
    bf_cases.to_csv(cases_path, index=False)

print('saved:', cases_path)
bf_cases.head(20)


In [ ]:
bf_cases['code'].value_counts()


In [ ]:
types = {
    'detector_events': str,
    'error_events': str,
    'true_observables': str,
    'bposd_pred_observables': str,
    'tesseract_pred_observables': str,
}

bf_cases = pd.read_csv(_data_paths.output('ca_bf_cases_csv_19', noise_model=noise_model, d_rounds=d_rounds, shots=shots), dtype=types)
bs_cases = pd.read_csv(_data_paths.output('ca_bs_cases_csv_20', noise_model=noise_model, d_rounds=d_rounds, shots=shots), dtype=types)
bfts_cases = pd.read_csv(_data_paths.output('ca_bfts_cases_csv_21', noise_model=noise_model, d_rounds=d_rounds, shots=shots), dtype=types)
bstf_cases = pd.read_csv(_data_paths.output('ca_bstf_cases_csv_22', noise_model=noise_model, d_rounds=d_rounds, shots=shots), dtype=types)


In [ ]:
frac_bp = bf_cases.shape[0] / (bf_cases.shape[0] + bfts_cases.shape[0])
frac_tess = bf_cases.shape[0] / (bf_cases.shape[0] + bstf_cases.shape[0])
print(f'fraction of both fail cases among all bp fail samples: {frac_bp:.4f}')
print(f'fraction of both fail cases among all tess fail samples: {frac_tess:.4f}')


In [ ]:
bfts_cases[(bfts_cases['code'] == 'CA_9_2_5') & (bfts_cases['error_weight'] == 2)]


CA_5_2_3

### Compare retained error-weight distributions

Summarize each code's error-weight distribution within the saved outcome groups. Selection and caps affect the retained counts, so these plots do not directly estimate physical-noise LER.


In [ ]:
error_weight_df = pd.DataFrame({
    'bf_cases': bf_cases[bf_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
    'bs_cases': bs_cases[bs_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
    'bfts_cases': bfts_cases[bfts_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
    'bstf_cases': bstf_cases[bstf_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
}).fillna(0).astype(int)
comparison_df = error_weight_df.rename_axis(columns='case').stack().rename('count').reset_index()
comparison_df['percentage'] = comparison_df['count'] / comparison_df.groupby('case')['count'].transform('sum') * 100
comparison_df = comparison_df.pivot(index='error_weight', columns='case', values=['count', 'percentage']).swaplevel(axis=1).sort_index(axis=1, level=0)
comparison_df


In [ ]:
count_df = comparison_df.xs('count', axis=1, level=1)
percentage_df = comparison_df.xs('percentage', axis=1, level=1)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.suptitle('Error Weight Distribution for CA_5_2_3', fontsize=12)

count_df.plot(
    kind='bar',
    title='Count vs Error Weight',
    ylabel='Count',
    ax=axes[0],
)
percentage_df.plot(
    kind='bar',
    title='Percentage vs Error Weight',
    ylabel='Percentage (%)',
    ax=axes[1],
)


CA_9_2_5

In [ ]:
error_weight_df = pd.DataFrame({
    'bf_cases': bf_cases[bf_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
    'bs_cases': bs_cases[bs_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
    'bfts_cases': bfts_cases[bfts_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
    'bstf_cases': bstf_cases[bstf_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
}).fillna(0).astype(int)
comparison_df = error_weight_df.rename_axis(columns='case').stack().rename('count').reset_index()
comparison_df['percentage'] = comparison_df['count'] / comparison_df.groupby('case')['count'].transform('sum') * 100
comparison_df = comparison_df.pivot(index='error_weight', columns='case', values=['count', 'percentage']).swaplevel(axis=1).sort_index(axis=1, level=0)
comparison_df


In [ ]:
count_df = comparison_df.xs('count', axis=1, level=1)
percentage_df = comparison_df.xs('percentage', axis=1, level=1)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.suptitle('Error Weight Distribution for CA_9_2_5', fontsize=12)

count_df.plot(
    kind='bar',
    title='Count vs Error Weight',
    ylabel='Count',
    ax=axes[0],
)
percentage_df.plot(
    kind='bar',
    title='Percentage vs Error Weight',
    ylabel='Percentage (%)',
    ax=axes[1],
)
